# E1 · Files to Iceberg with Spark

Every morning a delivery partner drops a CSV file of the parcels it shipped the day before.
Your job as the data engineer: turn those files into **one proper table** that anyone can
query with SQL, from any tool (Spark, Trino, DuckDB, Superset).

Keep the lesson (`README.md` in this folder) open next to this notebook: it explains *why*
each step works the way it does. Run the cells from top to bottom with **Shift+Enter**.
Cells marked **YOUR TURN** need a small change from you first.

## Step 1 · Create the input files

`make_data.py` writes three small CSV files into `data/`, one per day. Nothing is downloaded:
the rows come from a fixed random seed, so everybody gets the same files.

In [ ]:
import make_data

make_data.main()

## Step 2 · Look at a raw file

Read one file with pandas, **as text** (`dtype=str`), to see what the partner really sent.

In [ ]:
import pandas as pd

raw = pd.read_csv("data/shipments_2026-03-01.csv", dtype=str)
print(len(raw), "rows")
print(raw.dtypes)
raw.head()

Every column is text (`str`; older pandas versions say `object`). A CSV file has **no types**: `12.50` is
just four characters until someone decides it is a number. Deciding the types is part of
your job, and the table you create in step 5 will remember them.

## Step 3 · Connect to Spark

`spark()` opens a **Spark Connect** session as you, with your own login token. Your notebook
is only a *client*: it sends instructions to the shared Spark server, which does the work and
talks to the catalog.

`namespace()` gives the name of your own namespace for this track, `eng_<your name>`.

In [ ]:
import sys

sys.path.insert(0, "../_shared")
from lakehouse import spark, whoami
from trackcheck import namespace

s = spark("E1 files to iceberg")
ns = namespace()
print("Spark", s.version, "| you are", whoami(), "| your namespace:", ns)

## Step 4 · Create your namespace

A **namespace** (Trino calls it a *schema*) is a folder for tables in the catalog.
`IF NOT EXISTS` makes the cell safe to run twice.

In [ ]:
s.sql(f"CREATE NAMESPACE IF NOT EXISTS lakehouse.{ns}")
s.sql("SHOW NAMESPACES IN lakehouse").show()

## Step 5 · Create the table (YOUR TURN)

The table definition fixes the column **types** and the **partitioning**. Most queries on
shipments ask about some days ("yesterday", "last week"), so we split the data by *day of
`shipped_at`*. Iceberg computes the day from the timestamp itself (*hidden partitioning*):
nobody has to add a separate "date" column, and queries that filter on `shipped_at` skip the
other days automatically.

**YOUR TURN:** replace the `-- TODO` line with `PARTITIONED BY (days(shipped_at))`, then run
the cell.

In [ ]:
table = f"lakehouse.{ns}.shipments"

s.sql(f"""
CREATE TABLE IF NOT EXISTS {table} (
    shipment_id BIGINT,
    orderkey    BIGINT,
    carrier     STRING,
    shipped_at  TIMESTAMP_NTZ,
    weight_kg   DOUBLE,
    status      STRING)
USING iceberg
-- TODO: partition by the day of shipped_at
""")
print(s.sql(f"SHOW CREATE TABLE {table}").first()[0])

Look for `PARTITIONED BY (days(shipped_at))` in the output. If it is missing, see
"Common mistakes" in the lesson.

## Step 6 · Load the first file

Two things to notice:

1. **The Spark server cannot see your home folder.** `spark.read.csv("data/...")` would look
   for the file on the *server*, and fail with "Path does not exist". So we read the file
   here with pandas, convert the text to real types, and hand the rows to Spark with
   `createDataFrame`. (In production, files usually land in object storage first, where the
   cluster can read them directly.)
2. `writeTo(table).append()` adds the rows as **one commit**. Every commit makes a new
   *snapshot* of the table.

In [ ]:
def load_file(path):
    """Read one CSV file, give each column its real type, append it to the table."""
    pdf = pd.read_csv(path, dtype=str)
    pdf = pdf.astype({"shipment_id": "int64", "orderkey": "int64", "weight_kg": "float64"})
    pdf["shipped_at"] = pd.to_datetime(pdf["shipped_at"])
    df = s.createDataFrame(pdf, schema=s.table(table).schema)
    df.writeTo(table).append()
    return len(pdf)


n = load_file("data/shipments_2026-03-01.csv")
print("loaded", n, "rows; the table now has", s.table(table).count(), "rows")

## Step 7 · What did Iceberg write?

An Iceberg table is data files (Parquet) **plus** metadata that lists them. You can query the
metadata like a table: add `.snapshots`, `.files` or `.partitions` to the table name.

In [ ]:
s.sql(f"SELECT snapshot_id, operation, summary['added-records'] AS added_rows FROM {table}.snapshots").show(truncate=False)
s.sql(f"SELECT partition, record_count, file_count FROM {table}.partitions").show(truncate=False)

## Step 8 · Load the other two days (YOUR TURN)

**YOUR TURN:** call `load_file(...)` for `data/shipments_2026-03-02.csv` and
`data/shipments_2026-03-03.csv`, **once each**. Then run the cell. You should end with 1200
rows, 3 snapshots and 3 partitions.

In [ ]:
# TODO: load the files of 2026-03-02 and 2026-03-03 here


print("the table now has", s.table(table).count(), "rows (want 1200)")
s.sql(f"SELECT partition, record_count FROM {table}.partitions ORDER BY partition").show(truncate=False)

## Step 9 · Query it, with Spark and with Trino

The table lives in the **catalog**, not in Spark. Any engine that talks to the catalog sees the
same table. First Spark:

In [ ]:
s.sql(f"""
    SELECT carrier, count(*) AS parcels, round(avg(weight_kg), 1) AS avg_kg
    FROM {table}
    GROUP BY carrier
    ORDER BY parcels DESC""").show()

Now Trino, as you, joining your new table with the sample orders it was made for:

In [ ]:
from lakehouse import trino_connection

cur = trino_connection().cursor()
cur.execute(f"""
    SELECT o.orderpriority, count(*) AS parcels
    FROM lakehouse.{ns}.shipments AS sh
    JOIN lakehouse.samples.orders AS o ON o.orderkey = sh.orderkey
    GROUP BY o.orderpriority
    ORDER BY o.orderpriority""")
for row in cur.fetchall():
    print(row)

## Step 10 · Check your work

Run the checkpoint. It looks at your **table**, not at this notebook. You can also run
`lab-tracks check E1` in a terminal.

In [ ]:
!python checkpoint.py

When you are done, close your Spark session. It frees your slot on the shared server.

In [ ]:
s.stop()